# kob — one dataset, every transport

**kob** discovers a folder of Parquet and serves it as **Apache Arrow**. The product's
fast, default data path is **Arrow Flight** (gRPC); kob also ships *demo* servers for the
alternative wire formats so you can see the difference for yourself.

This notebook spins all of them up, runs the **same query** over each, checks they return
identical data, and times them:

| transport | what it is | port |
|---|---|---|
| Arrow **Flight** (gRPC) | the fast **default** data path | 8815 |
| Arrow **IPC over HTTP** *(demo)* | Arrow without gRPC | 8001 |
| **REST / JSON** *(demo)* | the row-oriented baseline | 8002 |
| **gRPC + Protobuf** *(demo)* | columnar protobuf | 8816 |
| **Swagger / HTTP** | discovery + interactive query | 8000 |

> **Run all cells top-to-bottom.** The first cell starts the servers; the last cell stops
> them. Install the deps with `uv sync --extra demo` and launch with
> `uv run --extra demo jupyter lab`.

## 1. Start the servers

`kob.server.app` gives us both Flight (`:8815`) and the Swagger control plane (`:8000`); each demo transport is its own little server.

In [ ]:
import json, socket, subprocess, sys, time
from pathlib import Path

import pandas as pd
import requests

# Find the repo root (so this runs from notebooks/ or the repo root alike).
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent

HOST = "127.0.0.1"
PORTS = dict(swagger=8000, arrow=8001, json=8002, flight=8815, proto=8816)

# Make sure there is sample data to serve.
if not (REPO / "data" / "optionmetrics").exists():
    print("generating sample data (one-off) ...")
    subprocess.run([sys.executable, "-m", "kob.tools.generate_data", "--scale", "small"],
                   cwd=REPO, check=True)

PROCS = []
def track(p): PROCS.append(p); return p

# The product: Arrow Flight (:8815) + Swagger HTTP (:8000), one process.
track(subprocess.Popen(
    [sys.executable, "-m", "kob.server.app", "--host", HOST,
     "--flight-port", str(PORTS["flight"]), "--http-port", str(PORTS["swagger"]), "--threads", "4"],
    cwd=REPO, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL))

# The demo transports, each standalone.
for module, port in [("kob.demos.arrow_http.server", PORTS["arrow"]),
                     ("kob.demos.json.server", PORTS["json"]),
                     ("kob.demos.protobuf.server", PORTS["proto"])]:
    track(subprocess.Popen(
        [sys.executable, "-m", module, "--host", HOST, "--port", str(port), "--threads", "4"],
        cwd=REPO, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL))

def wait_http(url, timeout=60):
    end = time.time() + timeout
    while time.time() < end:
        try:
            if requests.get(url, timeout=1).ok: return
        except requests.RequestException: time.sleep(0.3)
    raise TimeoutError(url)

def wait_tcp(port, timeout=60):
    end = time.time() + timeout
    while time.time() < end:
        try:
            with socket.create_connection((HOST, port), timeout=1): return
        except OSError: time.sleep(0.3)
    raise TimeoutError(port)

wait_http(f"http://{HOST}:{PORTS['swagger']}/health")
wait_http(f"http://{HOST}:{PORTS['arrow']}/health")
wait_http(f"http://{HOST}:{PORTS['json']}/health")
wait_tcp(PORTS["flight"])   # Flight is part of kob.server.app
wait_tcp(PORTS["proto"])
time.sleep(0.5)             # gRPC servers want a beat after the port opens
print("servers up:", PORTS)

## 2. Discover what's available (Swagger / HTTP)

No schema is hardcoded — kob derives everything from the folder layout. Ask it what exists and what you can filter on, *per folder* (partitions) and *per column* (facets).

In [ ]:
SWAGGER = f"http://{HOST}:{PORTS['swagger']}"

datasets = requests.get(f"{SWAGGER}/datasets").json()["datasets"]
pd.DataFrame([{"dataset": d["name"],
               "partitions (filter per folder)": [p["name"] for p in d["partition_columns"]],
               "data columns": len(d["data_columns"])} for d in datasets])

In [ ]:
# Partition values you can filter on per folder ...
ds = requests.get(f"{SWAGGER}/datasets/optionmetrics").json()
for p in ds["partition_columns"]:
    vals = p["values"]
    print(f"  {p['name']:>10}: {vals[:8]}{' ...' if len(vals) > 8 else ''}")

# ... and per-column min/max + approx-distinct (facets, from row-group stats).
facets = requests.get(f"{SWAGGER}/datasets/optionmetrics/facets",
                      params={"columns": "strike,delta,impl_vol", "distinct": "true"}).json()["facets"]
pd.DataFrame(facets).T

> Tip: open **http://127.0.0.1:8000/docs** in a browser for the full interactive Swagger UI, including a `/flight` page describing how to pull data over Flight.

## 3. One query, every transport

A single small JSON contract is understood by every transport. We'll ask for AAPL call options from 2023 with delta ≥ 0.4.

In [ ]:
from kob.core.contract import QueryRequest, Filter
from kob.tools.client import query_flight, query_http

QUERY = QueryRequest(
    dataset="optionmetrics",
    columns=["date", "strike", "cp_flag", "impl_vol", "delta", "und_price"],
    filters=[
        Filter("underlying", "=", "AAPL"),
        Filter("year", "=", 2023),
        Filter("cp_flag", "=", "C"),
        Filter("delta", ">=", 0.4),
    ],
)
print(json.dumps(QUERY.to_dict(), indent=2))

### Arrow Flight — the fast default ⚡

The Flight payload *is* the Arrow columnar layout: there is essentially nothing to deserialize on the client.

In [ ]:
flight_tbl = query_flight(f"grpc://{HOST}:{PORTS['flight']}", QUERY)
print(f"Flight: {flight_tbl.num_rows:,} rows x {flight_tbl.num_columns} cols")
flight_tbl.slice(0, 5).to_pandas()

### Arrow IPC over HTTP *(demo)*

The same Arrow bytes, streamed over plain HTTP instead of gRPC.

In [ ]:
arrow_http_tbl = query_http(f"http://{HOST}:{PORTS['arrow']}", QUERY)
print(f"HTTP-Arrow: {arrow_http_tbl.num_rows:,} rows")
arrow_http_tbl.slice(0, 5).to_pandas()

### REST / JSON *(demo)*

The ubiquitous baseline: row objects of text. The server encodes every value; the client decodes every value.

In [ ]:
rows = requests.post(f"http://{HOST}:{PORTS['json']}/query", json=QUERY.to_dict(), timeout=600).json()
json_df = pd.DataFrame(rows)
print(f"JSON: {len(json_df):,} rows")
json_df.head()

### gRPC + Protobuf *(demo, columnar)*

Protobuf is binary, but it is still a *messaging* format — every value is tag-encoded and must be parsed back into objects. Below we reconstruct the columns by hand, which is exactly the work Arrow avoids.

In [ ]:
import grpc
from kob.demos.protobuf.proto import data_service_pb2 as pb
from kob.demos.protobuf.proto import data_service_pb2_grpc as pbg

def query_proto(host, port, req, encoding="columnar"):
    channel = grpc.insecure_channel(
        f"{host}:{port}", options=[("grpc.max_receive_message_length", 512 * 1024 * 1024)])
    stub = pbg.DataServiceStub(channel)
    request = pb.QueryRequest(request_json=json.dumps(req.to_dict()), encoding=encoding)
    fields, cols = None, {}
    for reply in stub.Query(request):
        which = reply.WhichOneof("payload")
        if which == "schema":
            fields = [(f.name, f.type) for f in reply.schema.fields]
            cols = {name: [] for name, _ in fields}
        elif which == "col_batch":
            for (name, ptype), col in zip(fields, reply.col_batch.columns):
                if ptype == "double":  cols[name] += list(col.doubles)
                elif ptype == "int64": cols[name] += list(col.ints)
                elif ptype == "bool":  cols[name] += list(col.bools)
                else:                  cols[name] += list(col.strings)
    channel.close()
    return pd.DataFrame(cols)

proto_df = query_proto(HOST, PORTS["proto"], QUERY, "columnar")
print(f"Protobuf (columnar): {len(proto_df):,} rows")
proto_df.head()

### Same data, every time

The transport is just the wire — the result is identical.

In [ ]:
counts = {"flight": flight_tbl.num_rows, "http_arrow": arrow_http_tbl.num_rows,
          "json": len(json_df), "proto": len(proto_df)}
print("row counts:", counts)
assert len(set(counts.values())) == 1, "row counts differ across transports!"
print(f"✓ every transport returned the same {flight_tbl.num_rows:,} rows")

## 4. Why Flight is the default — a quick timing

End-to-end client wall time for the same query: issue → receive → decode into the native form. The row-oriented baselines are measured once (they're slow); the columnar ones take the median of a few runs.

In [ ]:
import statistics

def timeit(fn, reps):
    fn()                              # warmup
    ts = []
    for _ in range(reps):
        t = time.perf_counter(); fn(); ts.append(time.perf_counter() - t)
    return statistics.median(ts)

def time_once(fn):
    t = time.perf_counter(); fn(); return time.perf_counter() - t

timings = {
    "flight":     timeit(lambda: query_flight(f"grpc://{HOST}:{PORTS['flight']}", QUERY), 3),
    "http_arrow": timeit(lambda: query_http(f"http://{HOST}:{PORTS['arrow']}", QUERY), 3),
    "proto_col":  time_once(lambda: query_proto(HOST, PORTS["proto"], QUERY, "columnar")),
    "json":       time_once(lambda: requests.post(
                      f"http://{HOST}:{PORTS['json']}/query", json=QUERY.to_dict(), timeout=600).json()),
}

bench = pd.DataFrame({"transport": list(timings), "median_ms": [v * 1000 for v in timings.values()]})
flight_ms = bench.loc[bench.transport == "flight", "median_ms"].iloc[0]
bench["x_slower_than_flight"] = (bench["median_ms"] / flight_ms).round(1)
bench.sort_values("median_ms").reset_index(drop=True)

In [ ]:
try:
    import matplotlib.pyplot as plt
    ax = (bench.sort_values("median_ms").set_index("transport")["median_ms"]
          .plot.barh(logx=True, figsize=(7, 3), color=["#2a9d8f", "#264653", "#e9c46a", "#e76f51"]))
    ax.set_title("Median latency per transport (log scale — lower is better)")
    ax.set_xlabel("milliseconds")
    plt.tight_layout(); plt.show()
except ModuleNotFoundError:
    print("matplotlib not installed (install the [demo] extra) — see the table above.")

## 5. Shut the servers down

In [ ]:
for p in PROCS:
    p.terminate()
for p in PROCS:
    try: p.wait(timeout=5)
    except Exception: p.kill()
print("servers stopped.")

---

### Takeaways

* **Flight is the default because it's the fastest** — its wire format is the in-memory
  format, so the client does almost no work to read a result.
* The **HTTP-Arrow / JSON / Protobuf** servers exist only to *demonstrate* the alternatives;
  in production you'd reach for Flight (or the Arrow-over-HTTP demo if gRPC is inconvenient).
* Everything came from a folder of Parquet with **no schema configuration** — discovery,
  filtering and serialization are all kob's job.

More: [`README.md`](../README.md) · [`docs/DESIGN.md`](../docs/DESIGN.md) ·
[`docs/PERFORMANCE_REPORT.md`](../docs/PERFORMANCE_REPORT.md) (the full 5-way benchmark).